# Lesson 0: Setup and tensors

This notebook goes with the Lesson 0 page of the Forkcast course. Read the page first; the cells here are the same code, in order, ready to run.

Runtime: CPU is enough for this lesson. To see a GPU show up in the environment check, choose **Runtime → Change runtime type** and pick a GPU.

## 1. Clone your repository and install the package

Edit `GITHUB_USER` first. The clone fails with a clear message if the name is wrong.

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Environment check

This is the first definition-of-done command.

In [ ]:
!python -m forkcast.env

## 3. Meet tensors

For every cell, say the shape out loud *before* you run it.

In [ ]:
import torch
torch.manual_seed(0)                  # same random numbers every run

orders = torch.randn(32, 4)           # 32 orders, 4 features each
orders.shape, orders.dtype, orders.device

In [ ]:
print(orders[0].shape)        # the first order
print(orders[:, 1].shape)     # the second feature of every order
print(orders[:3, :2].shape)   # first three orders, first two features

In [ ]:
print(torch.tensor([1, 2, 3]).dtype)          # whole numbers give int64
print(torch.tensor([1.0, 2.0, 3.0]).dtype)    # the dtype models use
print(torch.zeros(2, 3).dtype)                # zeros/ones/randn default to float32

### Rearranging without changing the numbers

In [ ]:
one = orders[0]
print(one.unsqueeze(0).shape)            # a batch of one order

images = torch.zeros(8, 1, 28, 28)       # 8 grayscale images
print(images.flatten(1).shape)           # keep dim 0, merge the rest
print(images.permute(0, 2, 3, 1).shape)  # channels last
print(images.reshape(8, -1).shape)       # -1 means "work this one out"

x = torch.arange(6).reshape(2, 3)
print(x)
print(x.view(3, 2))        # same memory order, regrouped
print(x.permute(1, 0))     # rows become columns: a different reading order

### Broadcasting

The last cell reproduces the classic mistake on purpose, inside a `try` so the notebook keeps running.

In [ ]:
bias = torch.tensor([0.0, 1.0, 2.0, 3.0])       # one bias per feature
print((orders + bias).shape)                     # stretched over 32 orders

col_means = orders.mean(dim=0)                   # one mean per feature
print(col_means.shape, (orders - col_means).shape)

row_means = orders.mean(dim=1)                   # one mean per order, shape (32,)
try:
    orders - row_means
except RuntimeError as err:
    print("RuntimeError:", err)

row_means = orders.mean(dim=1, keepdim=True)     # shape (32, 1) broadcasts
print((orders - row_means).shape)

### Devices

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_orders = orders.to(device)       # .to() returns a new tensor; orders itself stays on the CPU
print(orders.device, gpu_orders.device)

## 4. Write the six functions

The cell below writes `forkcast/tensors.py` when you run it. Replace every `raise NotImplementedError(...)` with your code, run the cell to save the file, then run the tests in the next section. (You can also edit the file directly in Colab's file browser on the left, under `forkcast-course/forkcast/`.)

Hints are on the lesson page under step 6.

In [ ]:
%%writefile forkcast/tensors.py
"""Lesson 0. Six small tensor functions.

Each docstring states the exact input and output shapes the tests expect.
Replace every ``raise NotImplementedError`` with one to three lines of code.
Three of these come back later in the course: normalize_columns (Lesson 3),
flatten_images (Lesson 7) and pairwise_distance (Lesson 13).
"""

from __future__ import annotations

import torch
from torch import Tensor


def to_batch(x: Tensor) -> Tensor:
    """Turn one example into a batch of one.

    (d,) -> (1, d)
    """
    raise NotImplementedError("Lesson 0, step 6: add a dimension of size 1 at the front")


def flatten_images(images: Tensor) -> Tensor:
    """Flatten each image into one long vector, keeping the batch dimension.

    (N, C, H, W) -> (N, C*H*W)
    """
    raise NotImplementedError("Lesson 0, step 6: merge everything after dim 0")


def channels_last(images: Tensor) -> Tensor:
    """Move the channel dimension to the end.

    (N, C, H, W) -> (N, H, W, C)
    """
    raise NotImplementedError("Lesson 0, step 6: reorder the dimensions, batch stays first")


def normalize_columns(x: Tensor) -> Tensor:
    """Give every column mean 0 and standard deviation 1.

    (N, d) -> (N, d)
    Use the population standard deviation (``correction=0``).
    """
    raise NotImplementedError("Lesson 0, step 6: mean and std over dim 0, then broadcast")


def pairwise_distance(a: Tensor, b: Tensor) -> Tensor:
    """Euclidean distance between every row of ``a`` and every row of ``b``.

    (N, d), (M, d) -> (N, M)
    """
    raise NotImplementedError("Lesson 0, step 6: build an (N, M, d) difference tensor with broadcasting")


def move_to(x: Tensor, device: str | torch.device) -> Tensor:
    """Return ``x`` living on ``device`` ('cpu', 'cuda', or a torch.device)."""
    raise NotImplementedError("Lesson 0, step 6: one method call")


def one_hot(indices: Tensor, num_classes: int) -> Tensor:
    """Stretch goal. Class ids become one-hot rows.

    (N,) int64 -> (N, num_classes) float32, exactly one 1.0 per row.
    Do it with torch.arange, a comparison and broadcasting.
    """
    raise NotImplementedError("Lesson 0, stretch: compare indices[:, None] with torch.arange(num_classes)")

Quick manual check before the tests. Shapes first:

In [ ]:
import importlib
from forkcast import tensors
importlib.reload(tensors)   # pick up the file you just wrote

print(tensors.to_batch(torch.randn(4)).shape)                       # (1, 4)
print(tensors.flatten_images(torch.zeros(8, 3, 28, 28)).shape)      # (8, 2352)
print(tensors.channels_last(torch.zeros(8, 3, 28, 28)).shape)       # (8, 28, 28, 3)
print(tensors.normalize_columns(torch.randn(32, 4)).mean(dim=0))    # four numbers close to 0
print(tensors.pairwise_distance(torch.randn(5, 3), torch.randn(7, 3)).shape)   # (5, 7)
print(tensors.move_to(torch.randn(3), device).device)

## 5. Definition of done

Expected: `7 passed, 1 skipped, 1 deselected` on a CPU runtime, `8 passed, 1 deselected` on a GPU runtime (the deselected test is the stretch goal). A failing test prints the shape it got and the shape it expected.

In [ ]:
!pytest tests/test_00.py -q

### Stretch: `one_hot`

Optional. Build it from `torch.arange`, a comparison and broadcasting, then:

In [ ]:
!pytest tests/test_00.py -m stretch -q

## 6. Commit and push

Create a fine-grained personal access token on GitHub (Settings → Developer settings → Personal access tokens) for your `forkcast-course` repository with **Contents: read and write**. The cell asks for it with `getpass`, so it never appears in the notebook output. After the push, open the **Actions** tab of your repository: the *DoD checks* workflow runs the same fast tests and turns green.

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 0: tensors"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main